# Demographics / Baseline Clinical Table

This notebook creates the paper-style demographics table for the Prognosis task.

Output rule:
- Continuous variables: `mean ± SD` only.
- Categorical variables: `n (%)` only.
- Keep p values for label 0 vs label 1 within each dataset split.
- Do not output p-method or statistic columns.

All generated results are saved under `/host/d/projects/Habitats/results`.

In [1]:
# ============================================================
# 1. Imports
# ============================================================

import os
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)


In [2]:
# ============================================================
# 2. Settings
# ============================================================

patient_list_out_dir = '/host/e/D/Data/Habitats/Jishuitan/Patient_lists'
results_out_dir = '/host/d/projects/Habitats/results'
os.makedirs(results_out_dir, exist_ok=True)

processed_clinical_variables_path = os.path.join(
    patient_list_out_dir,
    'image_label_info_set123_clinical_variables_processed.xlsx',
)

split_file = os.path.join(
    patient_list_out_dir,
    'image_label_info_set123_5fold_prognosis_random0.xlsx',
)

demographics_table_path = os.path.join(
    results_out_dir,
    'demographics_table_prognosis.xlsx',
)

label_col = 'Prognosis_label'
lesion_site_levels = ['Femur', 'Tibia and fibula', 'Others']

print('processed_clinical_variables_path:', processed_clinical_variables_path)
print('split_file:', split_file)
print('demographics_table_path:', demographics_table_path)


processed_clinical_variables_path: /host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set123_clinical_variables_processed.xlsx
split_file: /host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set123_5fold_prognosis_random0.xlsx
demographics_table_path: /host/d/projects/Habitats/results/demographics_table_prognosis.xlsx


In [3]:
# ============================================================
# 3. Load processed clinical variables and split table
# ============================================================

clinical_df = pd.read_excel(
    processed_clinical_variables_path,
    dtype={'Patient_index': str},
)
clinical_df['Patient_set'] = clinical_df['Patient_set'].astype(str)
clinical_df['Patient_index'] = clinical_df['Patient_index'].astype(str)

split_df = pd.read_excel(
    split_file,
    dtype={'Patient_index': str},
)
split_df['Patient_set'] = split_df['Patient_set'].astype(str)
split_df['Patient_index'] = split_df['Patient_index'].astype(str)

split_keep_cols = ['Patient_set', 'Patient_index', 'split', 'fold', label_col]
missing_split_cols = [col for col in split_keep_cols if col not in split_df.columns]
if missing_split_cols:
    raise KeyError(f'Missing split columns: {missing_split_cols}')

clinical_for_merge = clinical_df.drop(columns=[label_col], errors='ignore').copy()

clinical_split_df = clinical_for_merge.merge(
    split_df[split_keep_cols],
    on=['Patient_set', 'Patient_index'],
    how='left',
)

if clinical_split_df['split'].isna().any():
    missing_cases = clinical_split_df.loc[
        clinical_split_df['split'].isna(),
        ['Patient_set', 'Patient_index'],
    ]
    raise RuntimeError(f'Some clinical cases were not found in split file:\n{missing_cases}')

print('clinical_split_df shape:', clinical_split_df.shape)
print('\nSplit counts:')
display(clinical_split_df['split'].value_counts(dropna=False))
print('\nLabel counts by split:')
display(pd.crosstab(clinical_split_df['split'], clinical_split_df[label_col]))


clinical_split_df shape: (351, 40)

Split counts:


train            188
internal test     98
external test     65
Name: split, dtype: int64


Label counts by split:


Prognosis_label,0,1
split,,
external test,45,20
internal test,69,29
train,132,56


In [4]:
# ============================================================
# 4. Variable lists
# ============================================================

categorical_variables_all = [
    'Sex',
    'Lesion_site',
    'Pathologic_fracture',
]

continuous_variables_all = [
    'Age',
    'Height_at_visit',
    'Weight_at_visit',
    'BMI',
    'WBC',
    'HGB',
    'PLT',
    'CRP',
    'ALP',
    'Total_cholesterol',
    'Triglycerides',
    'LDL',
    'LDH',
    'PT',
    'APTT',
    'Fibrinogen',
    'D_dimer',
    'Tumor_AP_diameter_mm',
    'Tumor_longitudinal_diameter_mm',
    'Tumor_transverse_diameter_mm',
    'Tumor_volume_mm3',
]

categorical_variables = [col for col in categorical_variables_all if col in clinical_split_df.columns]
continuous_variables = [col for col in continuous_variables_all if col in clinical_split_df.columns]

print('Categorical variables used:', categorical_variables)
print('Continuous variables used:', continuous_variables)


Categorical variables used: ['Sex', 'Lesion_site', 'Pathologic_fracture']
Continuous variables used: ['Age', 'Height_at_visit', 'Weight_at_visit', 'BMI', 'WBC', 'HGB', 'PLT', 'ALP', 'Total_cholesterol', 'Triglycerides', 'LDL', 'LDH', 'PT', 'Fibrinogen', 'D_dimer', 'Tumor_AP_diameter_mm', 'Tumor_longitudinal_diameter_mm', 'Tumor_transverse_diameter_mm', 'Tumor_volume_mm3']


In [5]:
# ============================================================
# 5. Helper functions
# ============================================================

def collapse_lesion_site_series(series):
    """Collapse lesion site into Femur, Tibia and fibula, or Others."""
    def _collapse_one(x):
        if pd.isna(x):
            return np.nan
        x = str(x).strip()
        x_lower = x.lower().replace('_', ' ').replace('-', ' ')
        x_lower = ' '.join(x_lower.split())
        if x_lower == 'femur':
            return 'Femur'
        if x_lower in ['tibia', 'fibula', 'tibia and fibula', 'tibia fibula', 'tibia/fibula']:
            return 'Tibia and fibula'
        return 'Others'
    return series.map(_collapse_one)


def format_p_value(p):
    if pd.isna(p):
        return ''
    p = float(p)
    if p < 0.001:
        return '<0.001'
    return f'{p:.3f}'


def format_mean_sd(values):
    values = pd.to_numeric(pd.Series(values), errors='coerce').dropna()
    if len(values) == 0:
        return 'NA'
    mean = values.mean()
    sd = values.std(ddof=1)
    return f'{mean:.3g} ± {sd:.3g}'


def continuous_p_value(x0, x1):
    """Same testing logic as clinical_features.ipynb, but only p value is reported."""
    x0 = pd.to_numeric(pd.Series(x0), errors='coerce').dropna()
    x1 = pd.to_numeric(pd.Series(x1), errors='coerce').dropna()
    if len(x0) < 2 or len(x1) < 2:
        return np.nan

    normal0 = stats.shapiro(x0).pvalue > 0.05 if 3 <= len(x0) <= 5000 else False
    normal1 = stats.shapiro(x1).pvalue > 0.05 if 3 <= len(x1) <= 5000 else False

    if normal0 and normal1:
        return float(stats.ttest_ind(x0, x1, equal_var=False, nan_policy='omit').pvalue)
    return float(stats.mannwhitneyu(x0, x1, alternative='two-sided').pvalue)


def values_equal_level(values, level):
    s = pd.Series(values).dropna()
    if len(s) == 0:
        return pd.Series([], dtype=bool)
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors='coerce') == float(level)
    return s.astype(str) == str(level)


def count_percent_summary(values, level):
    s = pd.Series(values).dropna()
    if len(s) == 0:
        return '0 (NA%)'
    n = int(values_equal_level(s, level).sum())
    pct = n / len(s) * 100.0
    return f'{n} ({pct:.1f}%)'


def categorical_global_p_value(values, group):
    """One global p value for a categorical variable across label 0/1."""
    tmp = pd.DataFrame({'value': values, 'group': group}).dropna()
    if tmp.shape[0] == 0 or tmp['group'].nunique() < 2 or tmp['value'].nunique() < 2:
        return np.nan

    table = pd.crosstab(tmp['value'].astype(str), tmp['group'].astype(int))
    table = table.reindex(columns=[0, 1], fill_value=0)
    chi2_stat, p_chi, dof, expected = stats.chi2_contingency(table.values)

    if table.shape == (2, 2) and np.any(expected < 5):
        _, p_fisher = stats.fisher_exact(table.values)
        return float(p_fisher)

    return float(p_chi)


In [6]:
# ============================================================
# 6. Build demographics table
# ============================================================

if 'Lesion_site' in clinical_split_df.columns:
    clinical_split_df['Lesion_site'] = collapse_lesion_site_series(clinical_split_df['Lesion_site'])

analysis_groups = {
    'training': clinical_split_df[clinical_split_df['split'] == 'train'].copy(),
    'internal_test': clinical_split_df[clinical_split_df['split'] == 'internal test'].copy(),
    'external_test': clinical_split_df[clinical_split_df['split'] == 'external test'].copy(),
}

for dataset_name, df_group in analysis_groups.items():
    print('\n============================================================')
    print('Dataset:', dataset_name, 'n =', df_group.shape[0])
    print(df_group[label_col].value_counts(dropna=False).sort_index())

demographic_rows = []


def add_continuous_row(variable, display_name=None):
    display_name = display_name or variable
    row = {
        'Variable': display_name,
        'Level': '',
        'Variable_type': 'continuous',
    }
    for dataset_name, df_group in analysis_groups.items():
        label0 = df_group[df_group[label_col] == 0]
        label1 = df_group[df_group[label_col] == 1]
        p = continuous_p_value(label0[variable], label1[variable])
        row[f'{dataset_name}_label0'] = format_mean_sd(label0[variable])
        row[f'{dataset_name}_label1'] = format_mean_sd(label1[variable])
        row[f'{dataset_name}_p_value'] = p
        row[f'{dataset_name}_p_value_formatted'] = format_p_value(p)
    demographic_rows.append(row)


def add_categorical_rows(variable, display_name, levels, level_display=None):
    if variable not in categorical_variables:
        return
    level_display = level_display or {level: str(level) for level in levels}

    row = {
        'Variable': display_name,
        'Level': '',
        'Variable_type': 'categorical',
    }
    for dataset_name, df_group in analysis_groups.items():
        p = categorical_global_p_value(df_group[variable], df_group[label_col])
        row[f'{dataset_name}_label0'] = ''
        row[f'{dataset_name}_label1'] = ''
        row[f'{dataset_name}_p_value'] = p
        row[f'{dataset_name}_p_value_formatted'] = format_p_value(p)
    demographic_rows.append(row)

    for level in levels:
        row = {
            'Variable': display_name,
            'Level': level_display.get(level, str(level)),
            'Variable_type': 'categorical_level',
        }
        for dataset_name, df_group in analysis_groups.items():
            label0_values = df_group.loc[df_group[label_col] == 0, variable]
            label1_values = df_group.loc[df_group[label_col] == 1, variable]
            row[f'{dataset_name}_label0'] = count_percent_summary(label0_values, level)
            row[f'{dataset_name}_label1'] = count_percent_summary(label1_values, level)
            row[f'{dataset_name}_p_value'] = ''
            row[f'{dataset_name}_p_value_formatted'] = ''
        demographic_rows.append(row)


add_categorical_rows(
    'Sex',
    'Sex',
    ['Female', 'Male'],
    {'Female': 'Female', 'Male': 'Male'},
)
add_categorical_rows(
    'Lesion_site',
    'Tumor location',
    lesion_site_levels,
    {'Femur': 'Femur', 'Tibia and fibula': 'Tibia and fibula', 'Others': 'Others'},
)
add_categorical_rows(
    'Pathologic_fracture',
    'Pathologic fracture',
    [0, 1],
    {0: 'No', 1: 'Yes'},
)

for var in continuous_variables:
    add_continuous_row(var)

demographics_df = pd.DataFrame(demographic_rows)

ordered_cols = ['Variable', 'Level', 'Variable_type']
for dataset_name in ['training', 'internal_test', 'external_test']:
    ordered_cols.extend([
        f'{dataset_name}_label0',
        f'{dataset_name}_label1',
        f'{dataset_name}_p_value',
        f'{dataset_name}_p_value_formatted',
    ])
demographics_df = demographics_df[[col for col in ordered_cols if col in demographics_df.columns]]

display(demographics_df)



Dataset: training n = 188
0    132
1     56
Name: Prognosis_label, dtype: int64

Dataset: internal_test n = 98
0    69
1    29
Name: Prognosis_label, dtype: int64

Dataset: external_test n = 65
0    45
1    20
Name: Prognosis_label, dtype: int64


,Variable,Level,Variable_type,training_label0,training_label1,training_p_value,training_p_value_formatted,internal_test_label0,internal_test_label1,internal_test_p_value,internal_test_p_value_formatted,external_test_label0,external_test_label1,external_test_p_value,external_test_p_value_formatted
0,Sex,,categorical,,,0.991773,0.992,,,0.920253,0.920,,,0.077831,0.078
1,Sex,Female,categorical_level,56 (42.4%),23 (41.1%),,,31 (44.9%),12 (41.4%),,,21 (46.7%),4 (20.0%),,
2,Sex,Male,categorical_level,76 (57.6%),33 (58.9%),,,38 (55.1%),17 (58.6%),,,24 (53.3%),16 (80.0%),,
3,Tumor location,,categorical,,,0.387092,0.387,,,0.667127,0.667,,,0.151438,0.151
4,Tumor location,Femur,categorical_level,69 (52.3%),34 (60.7%),,,34 (49.3%),16 (55.2%),,,22 (48.9%),12 (60.0%),,
5,Tumor location,Tibia and fibula,categorical_level,39 (29.5%),16 (28.6%),,,23 (33.3%),10 (34.5%),,,17 (37.8%),3 (15.0%),,
6,Tumor location,Others,categorical_level,24 (18.2%),6 (10.7%),,,12 (17.4%),3 (10.3%),,,6 (13.3%),5 (25.0%),,
7,Pathologic fracture,,categorical,,,0.237904,0.238,,,1.0,1.000,,,1.0,1.000
8,Pathologic fracture,No,categorical_level,119 (90.2%),54 (96.4%),,,65 (94.2%),28 (96.6%),,,44 (97.8%),20 (100.0%),,
9,Pathologic fracture,Yes,categorical_level,13 (9.8%),2 (3.6%),,,4 (5.8%),1 (3.4%),,,1 (2.2%),0 (0.0%),,


In [7]:
# ============================================================
# 7. Save demographics table
# ============================================================

demographics_df.to_excel(demographics_table_path, index=False)

print('Saved demographics table:')
print(demographics_table_path)
print('Shape:', demographics_df.shape)
demographics_df.head(30)


Saved demographics table:
/host/d/projects/Habitats/results/demographics_table_prognosis.xlsx
Shape: (29, 15)


,Variable,Level,Variable_type,training_label0,training_label1,training_p_value,training_p_value_formatted,internal_test_label0,internal_test_label1,internal_test_p_value,internal_test_p_value_formatted,external_test_label0,external_test_label1,external_test_p_value,external_test_p_value_formatted
0,Sex,,categorical,,,0.991773,0.992,,,0.920253,0.920,,,0.077831,0.078
1,Sex,Female,categorical_level,56 (42.4%),23 (41.1%),,,31 (44.9%),12 (41.4%),,,21 (46.7%),4 (20.0%),,
2,Sex,Male,categorical_level,76 (57.6%),33 (58.9%),,,38 (55.1%),17 (58.6%),,,24 (53.3%),16 (80.0%),,
3,Tumor location,,categorical,,,0.387092,0.387,,,0.667127,0.667,,,0.151438,0.151
4,Tumor location,Femur,categorical_level,69 (52.3%),34 (60.7%),,,34 (49.3%),16 (55.2%),,,22 (48.9%),12 (60.0%),,
5,Tumor location,Tibia and fibula,categorical_level,39 (29.5%),16 (28.6%),,,23 (33.3%),10 (34.5%),,,17 (37.8%),3 (15.0%),,
6,Tumor location,Others,categorical_level,24 (18.2%),6 (10.7%),,,12 (17.4%),3 (10.3%),,,6 (13.3%),5 (25.0%),,
7,Pathologic fracture,,categorical,,,0.237904,0.238,,,1.0,1.000,,,1.0,1.000
8,Pathologic fracture,No,categorical_level,119 (90.2%),54 (96.4%),,,65 (94.2%),28 (96.6%),,,44 (97.8%),20 (100.0%),,
9,Pathologic fracture,Yes,categorical_level,13 (9.8%),2 (3.6%),,,4 (5.8%),1 (3.4%),,,1 (2.2%),0 (0.0%),,


# Clinical Univariate and Multivariate Logistic Regression Table

This module is independent from the demographics table above: it reloads the processed clinical table and split file.

Output style follows a paper Table-3 pattern:
- one row per clinical variable;
- categorical variables are not split into one row per level;
- OR is reported with 95% CI;
- p value is reported for univariate and multivariate logistic regression.

For multi-category variables such as tumor location, the row-level p value is the global likelihood-ratio p value for the whole variable. The OR cell contains compact term-level comparisons against the reference group.


In [2]:
# ============================================================
# 8. Re-load data for logistic regression module
# ============================================================

import os
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import expit
from scipy.stats import norm, chi2

patient_list_out_dir = '/host/e/D/Data/Habitats/Jishuitan/Patient_lists'
results_out_dir = '/host/d/projects/Habitats/results'
os.makedirs(results_out_dir, exist_ok=True)

processed_clinical_variables_path = os.path.join(patient_list_out_dir, 'image_label_info_set123_clinical_variables_processed.xlsx')
split_file = os.path.join(patient_list_out_dir, 'image_label_info_set123_5fold_prognosis_random0.xlsx')
logistic_table_path = os.path.join(results_out_dir, 'clinical_univariate_multivariate_logistic_table_prognosis.xlsx')

label_col = 'Prognosis_label'
split_col = 'split'

# Report continuous-variable OR per 1 SD by default. This makes tumor-volume
# and laboratory variables numerically stable and easier to interpret.
continuous_or_scale = 'sd'  # choose: 'sd' or 'raw'
ridge_epsilon = 1e-6

clinical_df = pd.read_excel(processed_clinical_variables_path, dtype={'Patient_index': str})
clinical_df['Patient_set'] = clinical_df['Patient_set'].astype(str)
clinical_df['Patient_index'] = clinical_df['Patient_index'].astype(str)

split_df = pd.read_excel(split_file, dtype={'Patient_index': str})
split_df['Patient_set'] = split_df['Patient_set'].astype(str)
split_df['Patient_index'] = split_df['Patient_index'].astype(str)

split_keep_cols = ['Patient_set', 'Patient_index', split_col, 'fold', label_col]
missing_split_cols = [col for col in split_keep_cols if col not in split_df.columns]
if missing_split_cols:
    raise KeyError(f'Missing split columns: {missing_split_cols}')

clinical_for_merge = clinical_df.drop(columns=[label_col], errors='ignore').copy()
clinical_split_df = clinical_for_merge.merge(
    split_df[split_keep_cols],
    on=['Patient_set', 'Patient_index'],
    how='left',
)

if clinical_split_df[split_col].isna().any():
    missing_cases = clinical_split_df.loc[clinical_split_df[split_col].isna(), ['Patient_set', 'Patient_index']]
    raise RuntimeError(f'Some clinical cases were not found in split file:\n{missing_cases}')

train_df = clinical_split_df[clinical_split_df[split_col] == 'train'].copy()
train_df = train_df.dropna(subset=[label_col]).copy()
train_df[label_col] = train_df[label_col].astype(int)
y_train = train_df[label_col].to_numpy().astype(int)

print('processed_clinical_variables_path:', processed_clinical_variables_path)
print('split_file:', split_file)
print('logistic_table_path:', logistic_table_path)
print('Training n:', train_df.shape[0], 'events label=1:', int(y_train.sum()))
print('continuous_or_scale:', continuous_or_scale)


processed_clinical_variables_path: /host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set123_clinical_variables_processed.xlsx
split_file: /host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set123_5fold_prognosis_random0.xlsx
logistic_table_path: /host/d/projects/Habitats/results/clinical_univariate_multivariate_logistic_table_prognosis.xlsx
Training n: 188 events label=1: 56
continuous_or_scale: sd


In [3]:
# ============================================================
# 9. Logistic regression helpers
# ============================================================

def collapse_lesion_site_series(series):
    """Collapse lesion site into Femur, Tibia and fibula, or Others."""
    def _collapse_one(x):
        if pd.isna(x):
            return np.nan
        x = str(x).strip()
        x_lower = x.lower().replace('_', ' ').replace('-', ' ')
        x_lower = ' '.join(x_lower.split())
        if x_lower == 'femur':
            return 'Femur'
        if x_lower in ['tibia', 'fibula', 'tibia and fibula', 'tibia fibula', 'tibia/fibula']:
            return 'Tibia and fibula'
        return 'Others'
    return series.map(_collapse_one)

if 'Lesion_site' in train_df.columns:
    train_df['Lesion_site'] = collapse_lesion_site_series(train_df['Lesion_site'])

categorical_variables_all = ['Sex', 'Lesion_site', 'Pathologic_fracture']
continuous_variables_all = [
    'Age', 'Height_at_visit', 'Weight_at_visit', 'BMI',
    'WBC', 'HGB', 'PLT', 'CRP', 'ALP', 'Total_cholesterol', 'Triglycerides', 'LDL', 'LDH',
    'PT', 'APTT', 'Fibrinogen', 'D_dimer',
    'Tumor_AP_diameter_mm', 'Tumor_longitudinal_diameter_mm',
    'Tumor_transverse_diameter_mm', 'Tumor_volume_mm3',
]

categorical_variables = [col for col in categorical_variables_all if col in train_df.columns]
continuous_variables = [col for col in continuous_variables_all if col in train_df.columns]
logistic_variables = categorical_variables + continuous_variables

print('Categorical variables:', categorical_variables)
print('Continuous variables:', continuous_variables)
print('Number of clinical variables:', len(logistic_variables))

def null_log_likelihood(y):
    y = np.asarray(y, dtype=float)
    p = np.clip(y.mean(), 1e-12, 1 - 1e-12)
    return float(np.sum(y * np.log(p) + (1 - y) * np.log(1 - p)))

def prepare_design_matrix(df, variables, continuous_vars, categorical_vars, fit_info=None):
    """Prepare a numeric design matrix.

    Categorical handling:
    - Sex: Male vs Female.
    - Pathologic_fracture: Yes vs No.
    - Lesion_site: Femur and Tibia/fibula vs Others reference.

    Continuous handling:
    - default OR is per 1 SD, controlled by continuous_or_scale.
    """
    df = df.copy()
    X_parts = []
    term_to_variable = {}
    term_to_comparison = {}

    if fit_info is None:
        fit_info = {'continuous': {}, 'dummy_columns': {}}
        learn = True
    else:
        learn = False

    for var in variables:
        if var in continuous_vars:
            s = pd.to_numeric(df[var], errors='coerce').astype(float)
            if s.isna().any():
                raise RuntimeError(f'Unexpected missing value in processed continuous variable: {var}')
            if continuous_or_scale == 'sd':
                if learn:
                    mean = float(s.mean())
                    sd = float(s.std(ddof=1))
                    if not np.isfinite(sd) or sd == 0:
                        sd = 1.0
                    fit_info['continuous'][var] = {'mean': mean, 'sd': sd}
                mean = fit_info['continuous'][var]['mean']
                sd = fit_info['continuous'][var]['sd']
                s = (s - mean) / sd
                comparison = 'per 1 SD increase'
            elif continuous_or_scale == 'raw':
                comparison = 'per 1 unit increase'
            else:
                raise ValueError(f'Unknown continuous_or_scale: {continuous_or_scale}')
            X_parts.append(pd.DataFrame({var: s}, index=df.index))
            term_to_variable[var] = var
            term_to_comparison[var] = comparison
            continue

        if var == 'Sex':
            s = df[var].astype(str)
            col = 'Sex_Male'
            dummies = pd.DataFrame({col: (s == 'Male').astype(float)}, index=df.index)
            term_to_comparison[col] = 'Male vs Female'
        elif var == 'Pathologic_fracture':
            s = pd.to_numeric(df[var], errors='coerce')
            col = 'Pathologic_fracture_1'
            dummies = pd.DataFrame({col: (s == 1).astype(float)}, index=df.index)
            term_to_comparison[col] = 'Yes vs No'
        elif var == 'Lesion_site':
            s = collapse_lesion_site_series(df[var]).astype(str)
            dummies = pd.DataFrame({
                'Lesion_site_Femur': (s == 'Femur').astype(float),
                'Lesion_site_Tibia_and_fibula': (s == 'Tibia and fibula').astype(float),
            }, index=df.index)
            term_to_comparison['Lesion_site_Femur'] = 'Femur vs Others'
            term_to_comparison['Lesion_site_Tibia_and_fibula'] = 'Tibia and fibula vs Others'
        elif var in categorical_vars:
            s = df[var].astype(str)
            dummies = pd.get_dummies(s, prefix=var, drop_first=True, dtype=float)
            for col in dummies.columns:
                term_to_comparison[col] = col.replace(f'{var}_', '')
        else:
            raise KeyError(f'Variable {var} is neither continuous nor categorical.')

        if learn:
            fit_info['dummy_columns'][var] = list(dummies.columns)
        expected_cols = fit_info['dummy_columns'].get(var, list(dummies.columns))
        dummies = dummies.reindex(columns=expected_cols, fill_value=0.0)
        X_parts.append(dummies)
        for col in dummies.columns:
            term_to_variable[col] = var

    X = pd.concat(X_parts, axis=1) if len(X_parts) > 0 else pd.DataFrame(index=df.index)
    return X.astype(float), fit_info, term_to_variable, term_to_comparison

def fit_logistic_mle(X_df, y, ridge_epsilon=1e-6, maxiter=2000):
    X = np.asarray(X_df, dtype=float)
    y = np.asarray(y, dtype=float)
    X_const = np.column_stack([np.ones(X.shape[0]), X])
    names = ['Intercept'] + list(X_df.columns)

    def nll(beta):
        eta = X_const @ beta
        return np.sum(np.logaddexp(0, eta) - y * eta) + ridge_epsilon * np.sum(beta[1:] ** 2)

    result = minimize(nll, np.zeros(X_const.shape[1]), method='BFGS', options={'maxiter': maxiter})
    beta = result.x
    eta = X_const @ beta
    p = expit(eta)
    ll = np.sum(
        y * np.log(np.clip(p, 1e-12, 1 - 1e-12)) +
        (1 - y) * np.log(np.clip(1 - p, 1e-12, 1 - 1e-12))
    )

    W = p * (1 - p)
    information = X_const.T @ (X_const * W[:, None])
    information[1:, 1:] += np.eye(information.shape[0] - 1) * ridge_epsilon
    cov = np.linalg.pinv(information)
    se = np.sqrt(np.clip(np.diag(cov), 0, np.inf))
    z = beta / se
    p_values = 2 * (1 - norm.cdf(np.abs(z)))

    coef_df = pd.DataFrame({
        'Term': names,
        'Coef': beta,
        'SE': se,
        'Z': z,
        'P_value': p_values,
        'OR': np.exp(beta),
        'CI95_lower': np.exp(beta - 1.96 * se),
        'CI95_upper': np.exp(beta + 1.96 * se),
    })
    return {
        'success': bool(result.success),
        'message': str(result.message),
        'll': float(ll),
        'coef_df': coef_df,
        'n_params': X_const.shape[1],
    }

def format_p_value_for_table(p):
    if pd.isna(p):
        return ''
    p = float(p)
    if p < 0.001:
        return '<0.001'
    return f'{p:.3f}'

def format_or_ci(or_value, ci_low, ci_high):
    if pd.isna(or_value) or pd.isna(ci_low) or pd.isna(ci_high):
        return ''
    return f'{or_value:.3g} ({ci_low:.3g}, {ci_high:.3g})'

def compact_or_ci_for_variable(term_df, variable_name):
    rows = term_df[term_df['Variable'] == variable_name].copy()
    if rows.shape[0] == 0:
        return ''
    parts = []
    for _, row in rows.iterrows():
        orci = format_or_ci(row['OR'], row['CI95_lower'], row['CI95_upper'])
        comparison = row.get('Comparison', '')
        if comparison and comparison not in ['per 1 SD increase', 'per 1 unit increase']:
            parts.append(f'{comparison}: {orci}')
        else:
            parts.append(orci)
    return '\n'.join(parts)


Categorical variables: ['Sex', 'Lesion_site', 'Pathologic_fracture']
Continuous variables: ['Age', 'Height_at_visit', 'Weight_at_visit', 'BMI', 'WBC', 'HGB', 'PLT', 'ALP', 'Total_cholesterol', 'Triglycerides', 'LDL', 'LDH', 'PT', 'Fibrinogen', 'D_dimer', 'Tumor_AP_diameter_mm', 'Tumor_longitudinal_diameter_mm', 'Tumor_transverse_diameter_mm', 'Tumor_volume_mm3']
Number of clinical variables: 22


In [4]:
# ============================================================
# 10. Fit univariate and full multivariate logistic regression
# ============================================================

ll_null = null_log_likelihood(y_train)

univariate_variable_rows = []
univariate_term_rows = []

for var in logistic_variables:
    print('\n============================================================')
    print('Univariate logistic variable:', var)
    try:
        X_var, fit_info_var, term_to_variable, term_to_comparison = prepare_design_matrix(
            train_df, [var], continuous_variables, categorical_variables, fit_info=None,
        )
        if X_var.shape[1] == 0:
            raise RuntimeError('No usable columns after encoding.')
        if np.all(X_var.nunique(dropna=False) <= 1):
            raise RuntimeError('Variable has no variation after encoding.')

        fit = fit_logistic_mle(X_var, y_train, ridge_epsilon=ridge_epsilon)
        lr_stat = 2 * (fit['ll'] - ll_null)
        variable_p = float(chi2.sf(lr_stat, df=X_var.shape[1]))

        print('  encoded terms:', list(X_var.columns))
        print('  model success:', fit['success'], fit['message'])
        print('  variable likelihood-ratio p:', variable_p)

        coef_df = fit['coef_df'].copy()
        coef_df = coef_df[coef_df['Term'] != 'Intercept'].copy()
        coef_df.insert(0, 'Variable', var)
        coef_df.insert(1, 'Variable_LR_P_value', variable_p)
        coef_df.insert(2, 'Fit_success', fit['success'])
        coef_df.insert(3, 'Fit_message', str(fit['message']))
        coef_df['Comparison'] = coef_df['Term'].map(term_to_comparison)
        univariate_term_rows.append(coef_df)

        univariate_variable_rows.append({
            'Variable': var,
            'Variable_type': 'continuous' if var in continuous_variables else 'categorical',
            'Num_encoded_terms': X_var.shape[1],
            'LR_statistic': lr_stat,
            'Variable_p_value': variable_p,
            'Fit_success': fit['success'],
            'Fit_message': str(fit['message']),
        })
    except Exception as e:
        print('  Failed:', repr(e))
        univariate_variable_rows.append({
            'Variable': var,
            'Variable_type': 'continuous' if var in continuous_variables else 'categorical',
            'Num_encoded_terms': 0,
            'LR_statistic': np.nan,
            'Variable_p_value': np.nan,
            'Fit_success': False,
            'Fit_message': repr(e),
        })

univariate_variables_df = pd.DataFrame(univariate_variable_rows)
univariate_terms_df = pd.concat(univariate_term_rows, ignore_index=True) if len(univariate_term_rows) > 0 else pd.DataFrame()

# Full multivariate model: all clinical variables enter together.
X_multi, multi_fit_info, multi_term_to_variable, multi_term_to_comparison = prepare_design_matrix(
    train_df, logistic_variables, continuous_variables, categorical_variables, fit_info=None,
)
print('\n============================================================')
print('Full multivariate model')
print('Training cases:', len(y_train))
print('Events label=1:', int(y_train.sum()))
print('Encoded multivariate terms:', X_multi.shape[1])
print('Events per encoded term:', int(y_train.sum()) / max(X_multi.shape[1], 1))
if int(y_train.sum()) / max(X_multi.shape[1], 1) < 10:
    print('Warning: events per encoded term < 10. Multivariate estimates may be unstable.')

multi_fit = fit_logistic_mle(X_multi, y_train, ridge_epsilon=ridge_epsilon)
print('Multivariate fit success:', multi_fit['success'], multi_fit['message'])
multi_ll_full = multi_fit['ll']

multivariate_terms_df = multi_fit['coef_df'].copy()
multivariate_terms_df = multivariate_terms_df[multivariate_terms_df['Term'] != 'Intercept'].copy()
multivariate_terms_df['Variable'] = multivariate_terms_df['Term'].map(multi_term_to_variable)
multivariate_terms_df['Comparison'] = multivariate_terms_df['Term'].map(multi_term_to_comparison)
multivariate_terms_df.insert(1, 'Full_model_fit_success', multi_fit['success'])
multivariate_terms_df.insert(2, 'Full_model_fit_message', multi_fit['message'])

multivariate_variable_rows = []
for var in logistic_variables:
    reduced_vars = [v for v in logistic_variables if v != var]
    X_reduced, _, _, _ = prepare_design_matrix(
        train_df, reduced_vars, continuous_variables, categorical_variables, fit_info=None,
    )
    reduced_fit = fit_logistic_mle(X_reduced, y_train, ridge_epsilon=ridge_epsilon)
    df_diff = X_multi.shape[1] - X_reduced.shape[1]
    lr_stat = 2 * (multi_ll_full - reduced_fit['ll'])
    variable_p = float(chi2.sf(lr_stat, df=max(df_diff, 1)))
    multivariate_variable_rows.append({
        'Variable': var,
        'Variable_type': 'continuous' if var in continuous_variables else 'categorical',
        'Num_encoded_terms': int((multivariate_terms_df['Variable'] == var).sum()),
        'Adjusted_LR_statistic': lr_stat,
        'Adjusted_variable_p_value': variable_p,
        'Reduced_model_fit_success': reduced_fit['success'],
        'Reduced_model_fit_message': reduced_fit['message'],
    })

multivariate_variables_df = pd.DataFrame(multivariate_variable_rows)

display(univariate_variables_df.sort_values('Variable_p_value', na_position='last'))
display(multivariate_variables_df.sort_values('Adjusted_variable_p_value', na_position='last'))



Univariate logistic variable: Sex
  encoded terms: ['Sex_Male']
  model success: True Optimization terminated successfully.
  variable likelihood-ratio p: 0.8634645939060102

Univariate logistic variable: Lesion_site
  encoded terms: ['Lesion_site_Femur', 'Lesion_site_Tibia_and_fibula']
  model success: True Optimization terminated successfully.
  variable likelihood-ratio p: 0.36825218988274594

Univariate logistic variable: Pathologic_fracture
  encoded terms: ['Pathologic_fracture_1']
  model success: True Optimization terminated successfully.
  variable likelihood-ratio p: 0.11936571868114627

Univariate logistic variable: Age
  encoded terms: ['Age']
  model success: True Optimization terminated successfully.
  variable likelihood-ratio p: 0.6901243146342824

Univariate logistic variable: Height_at_visit
  encoded terms: ['Height_at_visit']
  model success: True Optimization terminated successfully.
  variable likelihood-ratio p: 0.3395308400152911

Univariate logistic variable: 

,Variable,Variable_type,Num_encoded_terms,LR_statistic,Variable_p_value,Fit_success,Fit_message
20,Tumor_transverse_diameter_mm,continuous,1,7.846573,0.005092,True,Optimization terminated successfully.
18,Tumor_AP_diameter_mm,continuous,1,6.097385,0.013538,True,Optimization terminated successfully.
12,Triglycerides,continuous,1,5.381337,0.020353,True,Optimization terminated successfully.
19,Tumor_longitudinal_diameter_mm,continuous,1,4.005807,0.045344,True,Optimization terminated successfully.
9,PLT,continuous,1,2.557922,0.109743,True,Optimization terminated successfully.
2,Pathologic_fracture,categorical,1,2.425624,0.119366,True,Optimization terminated successfully.
6,BMI,continuous,1,2.075058,0.149724,True,Optimization terminated successfully.
21,Tumor_volume_mm3,continuous,1,1.815786,0.177816,True,Optimization terminated successfully.
8,HGB,continuous,1,1.766196,0.183853,True,Optimization terminated successfully.
13,LDL,continuous,1,1.261762,0.261318,True,Optimization terminated successfully.


,Variable,Variable_type,Num_encoded_terms,Adjusted_LR_statistic,Adjusted_variable_p_value,Reduced_model_fit_success,Reduced_model_fit_message
21,Tumor_volume_mm3,continuous,1,3.302774,0.069163,True,Optimization terminated successfully.
15,PT,continuous,1,3.029983,0.081739,True,Optimization terminated successfully.
3,Age,continuous,1,2.957746,0.085467,True,Optimization terminated successfully.
20,Tumor_transverse_diameter_mm,continuous,1,2.841501,0.091858,True,Optimization terminated successfully.
12,Triglycerides,continuous,1,1.783325,0.181742,True,Optimization terminated successfully.
10,ALP,continuous,1,1.637112,0.200722,True,Optimization terminated successfully.
9,PLT,continuous,1,1.431074,0.231589,True,Optimization terminated successfully.
2,Pathologic_fracture,categorical,1,1.283610,0.257229,True,Optimization terminated successfully.
14,LDH,continuous,1,1.276969,0.258463,True,Optimization terminated successfully.
8,HGB,continuous,1,0.868184,0.351459,True,Optimization terminated successfully.


In [5]:
# ============================================================
# 11. Build and save paper-style logistic table
# ============================================================

paper_rows = []

for var in logistic_variables:
    var_type = 'continuous' if var in continuous_variables else 'categorical'
    uni_var_row = univariate_variables_df[univariate_variables_df['Variable'] == var]
    multi_var_row = multivariate_variables_df[multivariate_variables_df['Variable'] == var]

    uni_p = float(uni_var_row['Variable_p_value'].iloc[0]) if uni_var_row.shape[0] > 0 else np.nan
    multi_p = float(multi_var_row['Adjusted_variable_p_value'].iloc[0]) if multi_var_row.shape[0] > 0 else np.nan

    uni_or_ci = compact_or_ci_for_variable(univariate_terms_df, var)
    multi_or_ci = compact_or_ci_for_variable(multivariate_terms_df, var)

    if var_type == 'continuous':
        comparison = 'per 1 SD increase' if continuous_or_scale == 'sd' else 'per 1 unit increase'
    elif var == 'Sex':
        comparison = 'Male vs Female'
    elif var == 'Pathologic_fracture':
        comparison = 'Yes vs No'
    elif var == 'Lesion_site':
        comparison = 'Femur; Tibia and fibula vs Others'
    else:
        comparison = ''

    paper_rows.append({
        'Variable': var,
        'Variable_type': var_type,
        'Comparison_or_unit': comparison,
        'Univariate_OR_95CI': uni_or_ci,
        'Univariate_p_value': uni_p,
        'Univariate_p_value_formatted': format_p_value_for_table(uni_p),
        'Multivariate_OR_95CI': multi_or_ci,
        'Multivariate_p_value': multi_p,
        'Multivariate_p_value_formatted': format_p_value_for_table(multi_p),
    })

clinical_logistic_paper_table_df = pd.DataFrame(paper_rows)

with pd.ExcelWriter(logistic_table_path, engine='openpyxl') as writer:
    clinical_logistic_paper_table_df.to_excel(writer, sheet_name='paper_table', index=False)
    univariate_variables_df.to_excel(writer, sheet_name='univariate_variables', index=False)
    univariate_terms_df.to_excel(writer, sheet_name='univariate_terms', index=False)
    multivariate_variables_df.to_excel(writer, sheet_name='multivariate_variables', index=False)
    multivariate_terms_df.to_excel(writer, sheet_name='multivariate_terms', index=False)
    pd.DataFrame([{'continuous_or_scale': continuous_or_scale, 'ridge_epsilon': ridge_epsilon}]).to_excel(
        writer, sheet_name='settings', index=False
    )

print('Saved clinical logistic table:', logistic_table_path)
print('Shape:', clinical_logistic_paper_table_df.shape)
display(clinical_logistic_paper_table_df)


Saved clinical logistic table: /host/d/projects/Habitats/results/clinical_univariate_multivariate_logistic_table_prognosis.xlsx
Shape: (22, 9)


,Variable,Variable_type,Comparison_or_unit,Univariate_OR_95CI,Univariate_p_value,Univariate_p_value_formatted,Multivariate_OR_95CI,Multivariate_p_value,Multivariate_p_value_formatted
0,Sex,categorical,Male vs Female,"Male vs Female: 1.06 (0.561, 1.99)",0.863465,0.863,"Male vs Female: 1.15 (0.469, 2.85)",0.754347,0.754
1,Lesion_site,categorical,Femur; Tibia and fibula vs Others,"Femur vs Others: 1.97 (0.737, 5.27)\nTibia and...",0.368252,0.368,"Femur vs Others: 2.1 (0.594, 7.44)\nTibia and ...",0.494271,0.494
2,Pathologic_fracture,categorical,Yes vs No,"Yes vs No: 0.339 (0.0739, 1.55)",0.119366,0.119,"Yes vs No: 0.417 (0.0821, 2.12)",0.257229,0.257
3,Age,continuous,per 1 SD increase,"1.06 (0.785, 1.44)",0.690124,0.690,"1.47 (0.953, 2.26)",0.085467,0.085
4,Height_at_visit,continuous,per 1 SD increase,"1.17 (0.846, 1.61)",0.339531,0.340,"0.987 (0.247, 3.95)",0.985774,0.986
5,Weight_at_visit,continuous,per 1 SD increase,"0.93 (0.678, 1.28)",0.653944,0.654,"1.36 (0.0962, 19.1)",0.821949,0.822
6,BMI,continuous,per 1 SD increase,"0.789 (0.569, 1.1)",0.149724,0.150,"0.624 (0.0988, 3.95)",0.617815,0.618
7,WBC,continuous,per 1 SD increase,"0.961 (0.696, 1.33)",0.804918,0.805,"1.13 (0.769, 1.65)",0.546735,0.547
8,HGB,continuous,per 1 SD increase,"0.809 (0.59, 1.11)",0.183853,0.184,"0.83 (0.56, 1.23)",0.351459,0.351
9,PLT,continuous,per 1 SD increase,"0.76 (0.535, 1.08)",0.109743,0.110,"0.785 (0.523, 1.18)",0.231589,0.232
